# NHANES 2021-2023 Exploratory Data Analysis

⚠️ **Note**: This notebook contains original exploratory data analysis for the **2021-2023 NHANES cycle only**.

This analysis validates that the data matches NHANES documentation and explores feature distributions.

For production data preparation that processes **both cycles** (2021-2023 and 2017-2020), use `nhanes_data_preparation.ipynb`.

# Libraries

# Aux functions

In [ ]:
from collections.abc import Sequence

import matplotlib.pyplot as plt
import pandas as pd


def check_nans(df: pd.DataFrame, columns: Sequence[str],
                show_percent: bool = True,  # noqa: FBT001, FBT002
                sort: bool = True) -> pd.DataFrame:  # noqa: FBT001, FBT002
    """
    Quick check for NaNs in dataframe columns.
    Args:
        df: pandas.DataFrame
        columns: list of column names to check (None -> all columns)
        show_percent: include missing percentage column
        sort: sort result by missing_count desc
    Returns:
        pandas.DataFrame with index = column and columns:
            missing_count, missing_pct (optional), non_missing_count, dtype
    """  # noqa: DOC201

    if columns is None:  # type: ignore  # noqa: PGH003
        columns = list(df.columns)  # two spaces before comment
    total = len(df)  # two spaces before comment
    rows = []
    for col in columns:
        ser = df[col]  # two spaces before comment
        missing = int(ser.isna().sum())  # two spaces before comment

        rows.append({  # type: ignore  # noqa: PGH003
            "column": col,
            "missing_count": missing,
            "missing_pct": (missing / total) * 100 if total > 0 else 0.0,
            "non_missing_count": total - missing,
            "dtype": str(ser.dtype)
        })
    out = pd.DataFrame(rows).set_index("column")
    if not show_percent:
        out = out.drop(columns="missing_pct")
    if sort:
        out = out.sort_values("missing_count", ascending=False)
    return out

In [ ]:
def plot_value_counts(df: pd.DataFrame, column_name: str, xlabel: str | None = None, ylabel: str = "Frequency") -> None:  # type: ignore  # noqa: E501, PGH003
    """
    Plots a bar chart of value counts for a specified column in a DataFrame.

    Parameters:
        df (pd.DataFrame): The DataFrame containing the data.
        column_name (str): The column name to plot value counts for.
        xlabel (str | None, optional): Label for the x-axis. Defaults to the column name
        ylabel (str, optional): Label for the y-axis. Defaults to 'Frequency'.
    """
    counts = df[column_name].value_counts().sort_index()

    plt.figure(figsize=(10, 6))  # type: ignore  # noqa: PGH003
    bars = counts.plot(kind="bar", edgecolor="black")

    # Add value labels on top of each bar
    for container in bars.containers:
        bars.bar_label(container, fmt="%d")  # type: ignore  # noqa: PGH003

    plt.xlabel(xlabel or column_name)  # type: ignore  # noqa: PGH003
    plt.ylabel(ylabel)  # type: ignore  # noqa: PGH003
    plt.xticks(rotation=0)  # type: ignore  # noqa: PGH003
    plt.grid(axis="y", alpha=0.3)  # type: ignore  # noqa: PGH003
    plt.tight_layout()
    plt.show()  # type: ignore  # noqa: PGH003

# Data Loading (2021-2023 Cycle Only)

In [ ]:
# Load 2021-2023 NHANES cycle
df = pd.read_csv("../dataset/merged_data.csv", sep=";")  # type: ignore

# Feature Groups Definition

In [ ]:
# Demographics
DEMO_COLUMNS = {
    "CATEGORICAL": ["RIAGENDR", "RIDRETH3", "DMDEDUC2", "DMDMARTZ"],
    "CONTINUOUS": ["RIDAGEYR", "INDFMPIR", "DMDHHSIZ"]
}

# Body Measurements
BMX_COLUMNS = ["BMXBMI", "BMXWAIST", "BMXWT", "BMXHT"]

# Blood Pressure
BP_COLUMNS = ["BPXOSY1", "BPXOSY2", "BPXOSY3", "BPXODI1", "BPXODI2", "BPXODI3"]

# Smoking
SMQ_COLUMNS = ["SMQ020", "SMQ040", "SMD650"]

# Alcohol
ALQ_COLUMNS = ["ALQ121", "ALQ130", "ALQ170"]

# Physical Activity (2021-2023 format)
PAD_COLUMNS = ["PAD790Q", "PAD790U", "PAD800", "PAD810Q", "PAD810U", "PAD820", "PAD680"]

## Demographics

Categorical:

* RIAGENDR
* RIDRETH3
* DMDEDUC2 #Adults+20
* DMDMARTZ

Numerical:

* RIDAGEYR
* INDFMPIR
* DMDHHSIZ

### Numerical

In [ ]:
df[DEMO_COLUMNS["CONTINUOUS"]].describe()

In [ ]:
df[DEMO_COLUMNS["CONTINUOUS"]].info()

In [ ]:
df[DEMO_COLUMNS["CONTINUOUS"][1]].unique()

In [ ]:
# Let's investigate the INDFMPIR values more carefully

Nan checks

In [ ]:
check_nans(df, DEMO_COLUMNS["CONTINUOUS"])

### Categorical

In [ ]:
df[DEMO_COLUMNS["CATEGORICAL"]].describe()

In [ ]:
# RIAGENDR

Check Nans

In [ ]:
check_nans(df, DEMO_COLUMNS["CATEGORICAL"])

## Body Measurements

Continuous features:
 
* BMXBMI
* BMXWAIST
* BMXWT
* BMXHT  


In [ ]:
df[BMX_COLUMNS].describe()

In [ ]:
df[df["BMXWT"] < 45]  # noqa: PLR2004

Check Nans

In [ ]:
check_nans(df, BMX_COLUMNS)

## Blood Pressure

Continuous features:

* BPXOSY1 - Systolic reading 1
* BPXOSY2 - Systolic reading 2
* BPXOSY3 - Systolic reading 3
* BPXODI1 - Diastolic reading 1
* BPXODI2 - Diastolic reading 2
* BPXODI3 - Diastolic reading 3

In [ ]:
df[BP_COLUMNS].describe()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 10))  # type: ignore

# Plot systolic readings in the first column
axes[0, 0].hist(df["BPXOSY1"].dropna(), bins=30, edgecolor="black")
axes[0, 0].set_title("BPXOSY1 - Systolic Reading 1")
axes[0, 0].set_xlabel("Blood Pressure (mmHg)")
axes[0, 0].set_ylabel("Frequency")

axes[1, 0].hist(df["BPXOSY2"].dropna(), bins=30, edgecolor="black")
axes[1, 0].set_title("BPXOSY2 - Systolic Reading 2")
axes[1, 0].set_xlabel("Blood Pressure (mmHg)")
axes[1, 0].set_ylabel("Frequency")

axes[2, 0].hist(df["BPXOSY3"].dropna(), bins=30, edgecolor="black")
axes[2, 0].set_title("BPXOSY3 - Systolic Reading 3")
axes[2, 0].set_xlabel("Blood Pressure (mmHg)")
axes[2, 0].set_ylabel("Frequency")

# Plot diastolic readings in the second column
axes[0, 1].hist(df["BPXODI1"].dropna(), bins=30, edgecolor="black")
axes[0, 1].set_title("BPXODI1 - Diastolic Reading 1")
axes[0, 1].set_xlabel("Blood Pressure (mmHg)")
axes[0, 1].set_ylabel("Frequency")

axes[1, 1].hist(df["BPXODI2"].dropna(), bins=30, edgecolor="black")
axes[1, 1].set_title("BPXODI2 - Diastolic Reading 2")
axes[1, 1].set_xlabel("Blood Pressure (mmHg)")
axes[1, 1].set_ylabel("Frequency")

axes[2, 1].hist(df["BPXODI3"].dropna(), bins=30, edgecolor="black")
axes[2, 1].set_title("BPXODI3 - Diastolic Reading 3")
axes[2, 1].set_xlabel("Blood Pressure (mmHg)")
axes[2, 1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()  # type: ignore

In [ ]:
check_nans(df, BP_COLUMNS)

In [ ]:
# Check rows with missing values in all 3 systolic pressure measures
systolic_cols = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]
all_systolic_missing = df[systolic_cols].isna().all(axis=1).sum()

In [ ]:
# Check rows with missing values in all 3 diastolic pressure measures
diastolic_cols = ["BPXODI1", "BPXODI2", "BPXODI3"]
all_diastolic_missing = df[diastolic_cols].isna().all(axis=1).sum()

In [ ]:
# Check rows with missing values in ALL blood pressure measures (both systolic and diastolic)
all_bp_missing = df[BP_COLUMNS].isna().all(axis=1).sum()

## Smoking features

Continuous/Categorical features:

* SMQ020 - Ever smoked more than 100 cigarretes
* SMQ040 - Current smoking status [Categorical: 3 levels]
* SMD650 - Cigarettes per day [Continuous]


In [ ]:
plot_value_counts(df, "SMQ020", xlabel="Ever Smoked at least 100 Cigarettes")

### Smoking status [SMQ040]:

* 1 = Every day
* 2 = Some days
* 3 = not at all

In [ ]:
df["SMQ040"].describe()

In [ ]:
plot_value_counts(df, "SMQ040", xlabel="Current Smoking Status (SMQ040)")

### Cigarretes per day [SMD650]



In [ ]:
df["SMD650"].describe()

In [ ]:
smd650_col = df["SMD650"]

count_eq_1 = (smd650_col == 1).sum()
count_1_to_95 = ((smd650_col > 1) & (smd650_col <= 95)).sum()  # noqa: PLR2004
count_95_to_777 = ((smd650_col > 95) & (smd650_col < 777)).sum()  # noqa: PLR2004
count_777 = (smd650_col == 777).sum()  # noqa: PLR2004
count_999 = (smd650_col == 999).sum()  # noqa: PLR2004

print(f"Values equal to 1: {count_eq_1}")  # noqa: T201
print(f"Values between 1 and 95: {count_1_to_95}")  # noqa: T201
print(f"Values between 95 and 777: {count_95_to_777}")  # noqa: T201
print(f"Values equal to 777: {count_777}")  # noqa: T201
print(f"Values equal to 999: {count_999}")  # noqa: T201

In [ ]:
MAX_CIGARETTES = 95

plt.figure(figsize=(10, 6))  # type: ignore
plt.hist(df[(df["SMD650"] > 1) & (df["SMD650"] <= MAX_CIGARETTES)]["SMD650"].dropna(), bins=30, edgecolor="black")  # type: ignore
plt.xlabel("Cigarettes per day")  # type: ignore
plt.ylabel("Frequency")  # type: ignore
plt.title(f"Distribution of Cigarettes per Day (values between 1 and {MAX_CIGARETTES})")  # type: ignore
plt.grid(axis="y", alpha=0.3)  # type: ignore
plt.tight_layout()
plt.show()  # type: ignore

In [ ]:
check_nans(df, SMQ_COLUMNS)

## Alcohol consumption
Continuous/Categorical features:

* ALQ121 - Drinking frequency
* ALQ130 - Drinks per day
* ALQ170 - Binge drinking episodes on the last month


**ALQ121** -> Think how to group together some categories

![image.png](attachment:image.png)

**ALQ130** 

![image-2.png](attachment:image-2.png)

**ALQ170**

![image-3.png](attachment:image-3.png)

In [ ]:
for col in ALQ_COLUMNS:
    print(f"Value counts for {col}:")  # noqa: T201
    print(df[col].value_counts(dropna=False))  # noqa: T201
    print("-" * 40)  # noqa: T201

In [ ]:
check_nans(df, ALQ_COLUMNS)

## Physical Activity Features (2021-2023 Format)

Continuous/Categorical features:

* **PAD790Q** - Moderate activity frequency
* **PAD790U** - Moderate activity unit (D/W/M/Y) [Categorical]
* **PAD800** - Moderate activity minutes per session
* **PAD810Q** - Vigorous activity frequency
* **PAD810U** - Vigorous activity unit (D/W/M/Y) [Categorical]
* **PAD820** - Vigorous activity minutes per session
* **PAD680** - Sedentary minutes per day

In [ ]:
check_nans(df, PAD_COLUMNS)

### Frequency features

* **PAD790Q** - Moderate activity frequency
* **PAD810Q** - Vigorous activity frequency

![image.png](attachment:image.png)


![image-2.png](attachment:image-2.png)

In [ ]:
FREQ = ["PAD790Q", "PAD810Q"]

In [ ]:
df[FREQ].describe()

In [ ]:
df[FREQ].info()

In [ ]:
df[(df["PAD790Q"] > 4) & (df["PAD790Q"] < 7777)]  # noqa: PLR2004

In [ ]:
df[(df["PAD810Q"] > 4) & (df["PAD810Q"] < 7777)]  # noqa: PLR2004

In [ ]:
# Replace sentinel 9999 with NA for PAD790Q and PAD810Q
cols = ["PAD790Q", "PAD810Q"]

for col in cols:
    before_count = (df[col] == 9999).sum()
    df[col] = df[col].replace(9999, pd.NA)
    after_na = df[col].isna().sum()

### Unit Features

* **PAD790U** - Moderate activity unit (D/W/M/Y) [Categorical]
* **PAD810U** - Vigorous activity unit (D/W/M/Y) [Categorical]

In [ ]:
UNIT_FEATURES = ["PAD790U", "PAD810U"]

In [ ]:
plot_value_counts(df, "PAD790U", xlabel="Moderate Activity Unit (PAD790U)")
plot_value_counts(df, "PAD810U", xlabel="Vigorous Activity Unit (PAD810U)")

In [ ]:
# Transform unit features to numerical "every_X_days" representation
# D (Daily) -> 1 (frequency per 1 day)
# W (Weekly) -> 7 (frequency per 7 days)
# M (Monthly) -> 30 (frequency per 30 days)
# Y (Yearly) -> 365 (frequency per 365 days)

unit_to_days_mapping = {
    "D": 1,
    "W": 7,
    "M": 30,
    "Y": 365
}

# Get position of original unit columns to insert new features there
pad790u_pos = df.columns.get_loc("PAD790U")
pad810u_pos = df.columns.get_loc("PAD810U")

# Transform moderate activity unit
df.insert(pad790u_pos + 1, "moderate_every_X_days", df["PAD790U"].map(unit_to_days_mapping))  # type: ignore

# Transform vigorous activity unit (position adjusted because we inserted moderate_every_X_days)
df.insert(pad810u_pos + 2, "vigorous_every_X_days", df["PAD810U"].map(unit_to_days_mapping))  # type: ignore

# Drop original unit columns
df = df.drop(columns=["PAD790U", "PAD810U"])

### Minutes of activity features

* **PAD800** - Moderate activity minutes per session

![image.png](attachment:image.png)



* **PAD820** - Vigorous activity minutes per session

![image-2.png](attachment:image-2.png)

* **PAD680** - Sedentary minutes per day

![image-3.png](attachment:image-3.png)

In [ ]:
MIN_FEATURES = ["PAD800", "PAD820", "PAD680"]

In [ ]:
df[MIN_FEATURES].describe()

In [ ]:
df[df[MIN_FEATURES[0]] > 720]["PAD800"].value_counts()  # noqa: PLR2004

In [ ]:
df[df[MIN_FEATURES[1]] > 900]["PAD820"].value_counts()  # noqa: PLR2004

In [ ]:
df[df[MIN_FEATURES[2]] > 1380]["PAD680"].value_counts()  # noqa: PLR2004

In [ ]:
# Replace sentinel 9999 with NA
cols = ["PAD800", "PAD820", "PAD680"]

for col in cols:
    before_count = (df[col] == 9999).sum()
    df[col] = df[col].replace(9999, pd.NA)
    after_na = df[col].isna().sum()